In [22]:
from groq import Groq

client = Groq()

models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-120b
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
groq/compound-mini
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo


In [24]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="groq/compound-mini",
    temperature=0
)

response = llm.invoke(
    " hi response should be in english "
)
response.content

'Hello! How can I help you today?'

In [29]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# Subtask 
class Task(BaseModel):
    id:str
    title:str
    brief:str=Field(...,description="What to cover")


In [28]:
# Plan 
class Plan (BaseModel):
    blog_title:str
    tasks: List[Task]
    


In [31]:
from typing import final
from parso.python.tree import Operator
from pygments.token import Literal
# State 
class State(TypedDict):
    topic : str
    plan :Plan
    sections:Annotated[Literal[str],Operator.add]
    final:str

In [35]:
llm = ChatGroq(
    model="groq/compound-mini",
    temperature=0
)

In [36]:
def orchestrator(state: State) -> dict:

    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}

In [37]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]